# Task 3 — Quantized Encoders Report Companion

This notebook is a cleaned, report-aligned version of the earlier exploratory analysis.  
It is organized to mirror `3_report_quantized_encoders.pdf` and focuses on reproducible, showcase-friendly sections:

1. Annotation agreement
2. Annotation-to-label conversion
3. Label characteristics and co-occurrence
4. Metadata and audio feature analysis
5. Feature-space visualization

The notebook expects access to the MLPC development dataset, but the code is structured so paths can be adapted in one place.


## Setup and dataset paths

Place the MLPC development dataset under `data/MLPC2026_dataset_development/`. The notebook resolves all dataset paths relative to the repository root.


In [ ]:
from pathlib import Path
import math
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('default')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / 'task_3').exists():
    PROJECT_ROOT = NOTEBOOK_DIR
elif NOTEBOOK_DIR.name == 'notebooks' and NOTEBOOK_DIR.parent.name == 'task_3':
    PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
else:
    PROJECT_ROOT = NOTEBOOK_DIR

TASK3_DIR = PROJECT_ROOT / 'task_3'
REPORT_DIR = TASK3_DIR / 'report'
FIGURES_DIR = TASK3_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT = PROJECT_ROOT / 'data'
DATASET_DIR = DATA_ROOT / 'MLPC2026_dataset_development'
FEATURES_DIR = DATASET_DIR / 'audio_features'
METADATA_CSV = DATASET_DIR / 'metadata.csv'

print(f'Project root : {PROJECT_ROOT}')
print(f'Report dir   : {REPORT_DIR}')
print(f'Figures dir  : {FIGURES_DIR}')
print(f'Dataset dir  : {DATASET_DIR}')
print(f'Data root    : {DATA_ROOT}')
print(f'Dataset found: {DATASET_DIR.exists()}')


## Imports, metadata, and target classes


In [ ]:
metadata = pd.read_csv(METADATA_CSV)
metadata = metadata.copy()
metadata['filename'] = metadata['filename'].str.replace('.wav', '', regex=False)
metadata = metadata.set_index('filename')

TARGET_CLASSES = sorted([
    'keyboard_typing',
    'keychain',
    'footsteps',
    'door_open_close',
    'cutlery_dishes',
    'toilet_flushing',
    'running_water',
    'light_switch',
    'window_open_close',
    'bell_ringing',
    'wardrobe_drawer_open_close',
    'microwave',
    'vacuum_cleaner',
    'phone_ringing',
    'coffee_machine',
])
C = len(TARGET_CLASSES)

metadata.head()

In [ ]:
npz_files = sorted([path.name for path in FEATURES_DIR.glob('*.npz')])
feature_file_map = {
    path.stem: path
    for path in FEATURES_DIR.glob('*.npz')
}

print(f'Found {len(feature_file_map)} feature files.')
print(TARGET_CLASSES)

In [ ]:
def load_feature_file(file_id: str):
    path = feature_file_map[file_id]
    return np.load(path, allow_pickle=True)


def parse_class_list(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() in {'none', 'nan'}:
        return []
    return [item.strip() for item in text.split(';') if item.strip()]


def build_target_class_index_from_metadata():
    index = {target_class: [] for target_class in TARGET_CLASSES}
    for file_id, row in metadata.iterrows():
        if file_id not in feature_file_map:
            continue
        target_classes = parse_class_list(row.get('target_classes', ''))
        for target_class in target_classes:
            if target_class in index:
                index[target_class].append(file_id)
    return {key: pd.Index(value) for key, value in index.items()}


target_class_index = build_target_class_index_from_metadata()
metadata[['target_classes']].head()

## 1. Annotation verification and case-study context

The report includes a qualitative case study for `005699.wav`. The notebook keeps that section lightweight: it documents the file ID and uses the exported report figures for the visual audit, while the quantitative sections below remain fully reproducible.


In [ ]:
CASE_STUDY_FILE = '005699'
case_study_row = metadata.loc[[CASE_STUDY_FILE]] if CASE_STUDY_FILE in metadata.index else pd.DataFrame()
case_study_row

In [ ]:
for figure_name in ['1c_Ann1.png', '1c_Ann2.png', '1c_Ann3.png', '1c_all_Ann.png']:
    figure_path = FIGURES_DIR / figure_name
    if figure_path.exists():
        img = plt.imread(figure_path)
        plt.figure(figsize=(14, 4 if 'all' not in figure_name else 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(figure_name)
        plt.show()

## 2.1 Annotator agreement

Agreement is measured as frame-level Jaccard overlap between annotators. To match the report, class-specific agreement is computed over metadata-defined target-class subsets, not only files where a class was later detected as active in the annotation tensor.


In [ ]:
def agreement_rates_for_file_ids(file_ids) -> np.ndarray:
    rates = []
    for file_id in file_ids:
        data = load_feature_file(file_id)
        if 'annotations' not in data or data['annotations'].shape[2] < 2:
            continue
        binary = data['annotations'] > 0
        intersection = binary.all(axis=2).sum()
        union = binary.any(axis=2).sum()
        if union > 0:
            rates.append(intersection / union)
    return np.asarray(rates, dtype=float)


def agreement_rates_for_class(target_class: str) -> np.ndarray:
    return agreement_rates_for_file_ids(target_class_index[target_class])


agreement_dict = {
    target_class: agreement_rates_for_class(target_class)
    for target_class in TARGET_CLASSES
}

global_agreement_rates = agreement_rates_for_file_ids(sorted(feature_file_map))
global_agreement = float(global_agreement_rates.mean())

agreement_summary = pd.DataFrame([
    {
        'class': target_class,
        'mean_jaccard': values.mean(),
        'std_jaccard': values.std(),
        'min_jaccard': values.min(),
        'max_jaccard': values.max(),
        'n_recordings': len(values),
    }
    for target_class, values in agreement_dict.items()
    if len(values) > 0
]).sort_values('mean_jaccard', ascending=False)

class_mean_agreement = float(agreement_summary['mean_jaccard'].mean())
print(f'All-file Jaccard agreement: {global_agreement:.3f}')
print(f'Mean of class-specific Jaccard means: {class_mean_agreement:.3f}')
agreement_summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
plot_df = agreement_summary.sort_values('mean_jaccard', ascending=False)
colors = ['steelblue' if value >= class_mean_agreement else 'indianred' for value in plot_df['mean_jaccard']]

ax.bar(plot_df['class'], plot_df['mean_jaccard'], color=colors)
ax.axhline(class_mean_agreement, color='gray', linestyle='--', linewidth=1,
           label=f'Class-mean threshold = {class_mean_agreement:.3f}')
ax.axhline(global_agreement, color='black', linestyle=':', linewidth=1,
           label=f'All-file mean = {global_agreement:.3f}')
ax.set_title('Annotator Agreement by Class')
ax.set_ylabel('Jaccard agreement')
ax.set_ylim(0, 1)
ax.set_xticklabels(plot_df['class'], rotation=45, ha='right')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '2a_Annotator_Agreement.png', dpi=200, bbox_inches='tight')
plt.show()

## 2.2 Convert annotations to labels

The report uses annotator-mean aggregation followed by majority-vote binarization.


In [ ]:
def aggregate_labels(file_id: str):
    data = load_feature_file(file_id)
    if 'annotations' not in data or data['annotations'].shape[2] < 2:
        return None
    soft_labels = data['annotations'].mean(axis=2)
    hard_labels = (soft_labels >= 0.5).astype(int)
    return soft_labels, hard_labels


soft_labels_dict = {}
hard_labels_dict = {}
for file_id in feature_file_map:
    aggregated = aggregate_labels(file_id)
    if aggregated is None:
        continue
    soft_labels_dict[file_id], hard_labels_dict[file_id] = aggregated

print(f'Soft-label recordings: {len(soft_labels_dict)}')
print(f'Hard-label recordings: {len(hard_labels_dict)}')

In [ ]:
all_feature_file_ids = set(feature_file_map)
usable_label_file_ids = set(hard_labels_dict)
excluded_before_aggregation = sorted(all_feature_file_ids - usable_label_file_ids)

invalid_shape_files = []
for file_id, labels in hard_labels_dict.items():
    array = np.asarray(labels)
    if array.ndim != 2 or array.shape[1] != C:
        invalid_shape_files.append(file_id)

for file_id in invalid_shape_files:
    hard_labels_dict.pop(file_id, None)
    soft_labels_dict.pop(file_id, None)

print(f'Feature files available: {len(all_feature_file_ids)}')
print(f'Excluded before label aggregation: {len(excluded_before_aggregation)}')
print(f'Removed after shape validation: {len(invalid_shape_files)}')
print(f'Remaining recordings: {len(hard_labels_dict)}')

The report describes 625 recordings being removed before the final label dictionary is used. In this notebook, those recordings are excluded during aggregation because they do not provide a usable annotation tensor; the final usable count remains 3031 recordings.


## 2.3 Label characteristics

This section covers class frequencies, placement effects, and co-occurrence at frame level and file level.


In [ ]:
valid_file_ids = sorted(hard_labels_dict.keys())
metadata_clean = metadata.loc[metadata.index.intersection(valid_file_ids)].copy()
all_hard_labels = np.concatenate([hard_labels_dict[file_id] for file_id in metadata_clean.index], axis=0)
class_frequency = all_hard_labels.sum(axis=0)

label_frequency_df = pd.DataFrame({
    'class': TARGET_CLASSES,
    'active_frames': class_frequency,
}).sort_values('active_frames', ascending=False)

label_frequency_df

In [ ]:
order = label_frequency_df['class'].tolist()
order_idx = [TARGET_CLASSES.index(class_name) for class_name in order]

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(label_frequency_df['class'], label_frequency_df['active_frames'], color='steelblue')
ax.set_title('Label Frequency by Class on Frame Level')
ax.set_ylabel('Active frames')
ax.set_xticklabels(label_frequency_df['class'], rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
placement_frequency = {}
for placement, group_df in metadata_clean.groupby('device_placement'):
    arrays = [hard_labels_dict[file_id] for file_id in group_df.index]
    placement_frequency[placement] = np.concatenate(arrays, axis=0).sum(axis=0)

placements = list(placement_frequency.keys())
x = np.arange(C)
width = 0.8 / max(len(placements), 1)

fig, ax = plt.subplots(figsize=(12, 6))
for position, placement in enumerate(placements):
    values = placement_frequency[placement][order_idx]
    offset = (position - (len(placements) - 1) / 2) * width
    ax.bar(x + offset, values, width=width, label=placement)

ax.set_xticks(x)
ax.set_xticklabels(order, rotation=45, ha='right')
ax.set_ylabel('Active frames')
ax.set_title('Label Frequency by Class and Device Placement')
ax.legend(title='Device placement')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '2c_Fig_Label_Frequency_Placement.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
def frame_level_cooccurrence_matrix(label_dict: dict[str, np.ndarray]) -> np.ndarray:
    all_labels = np.concatenate([label_dict[file_id] for file_id in sorted(label_dict)], axis=0)
    return all_labels.T @ all_labels


def jaccard_from_binary_labels(label_dict: dict[str, np.ndarray]) -> np.ndarray:
    labels = np.concatenate([label_dict[file_id] for file_id in sorted(label_dict)], axis=0)
    counts = labels.sum(axis=0)
    cooc = labels.T @ labels
    result = np.zeros((C, C), dtype=float)
    for i in range(C):
        for j in range(C):
            union = counts[i] + counts[j] - cooc[i, j]
            result[i, j] = cooc[i, j] / union if union > 0 else 0.0
    np.fill_diagonal(result, np.nan)
    return result


def npmi_from_binary_labels(label_dict: dict[str, np.ndarray], mode: str = 'frame') -> np.ndarray:
    if mode == 'frame':
        labels = np.concatenate([label_dict[file_id] for file_id in sorted(label_dict)], axis=0)
    elif mode == 'file':
        labels = np.asarray([
            (label_dict[file_id].sum(axis=0) > 0).astype(float)
            for file_id in sorted(label_dict)
        ])
    else:
        raise ValueError("mode must be 'frame' or 'file'")

    p_class = labels.mean(axis=0)
    p_joint = (labels.T @ labels) / len(labels)
    p_outer = np.outer(p_class, p_class)
    with np.errstate(divide='ignore', invalid='ignore'):
        pmi = np.where(p_joint > 0, np.log(p_joint / p_outer), 0.0)
        npmi = np.where(p_joint > 0, pmi / -np.log(p_joint), 0.0)
    np.fill_diagonal(npmi, np.nan)
    return npmi


jaccard_frame = jaccard_from_binary_labels(hard_labels_dict)
npmi_frame = npmi_from_binary_labels(hard_labels_dict, mode='frame')
npmi_file = npmi_from_binary_labels(hard_labels_dict, mode='file')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, matrix, title, cmap, limits in [
    (axes[0], jaccard_frame, 'Co-occurrence (Jaccard), Frame Level', 'Blues', (0, 0.3)),
    (axes[1], npmi_frame, 'Co-occurrence (NPMI), Frame Level', 'RdBu_r', (-0.3, 0.3)),
]:
    image = ax.imshow(matrix, cmap=cmap, vmin=limits[0], vmax=limits[1])
    ax.set_title(title)
    ax.set_xticks(range(C))
    ax.set_xticklabels(TARGET_CLASSES, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(C))
    ax.set_yticklabels(TARGET_CLASSES, fontsize=8)
    plt.colorbar(image, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, matrix, title in [
    (axes[0], npmi_frame, 'Frame-level NPMI'),
    (axes[1], npmi_file, 'File-level NPMI'),
]:
    image = ax.imshow(matrix, cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    ax.set_title(title)
    ax.set_xticks(range(C))
    ax.set_xticklabels(TARGET_CLASSES, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(C))
    ax.set_yticklabels(TARGET_CLASSES, fontsize=8)
    plt.colorbar(image, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '2c_Fig_Cooccurrence_NPMI.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
mask = np.triu(np.ones((C, C), dtype=bool), k=1)
pairs = []
for i in range(C):
    for j in range(C):
        if mask[i, j] and not np.isnan(npmi_frame[i, j]) and not np.isnan(npmi_file[i, j]):
            pairs.append({
                'pair': f'{TARGET_CLASSES[i]} + {TARGET_CLASSES[j]}',
                'frame_npmi': float(npmi_frame[i, j]),
                'file_npmi': float(npmi_file[i, j]),
                'gap': float(npmi_file[i, j] - npmi_frame[i, j]),
            })

pairs_df = pd.DataFrame(pairs).sort_values('gap', ascending=False)
pairs_df.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(pairs_df['frame_npmi'], pairs_df['file_npmi'], alpha=0.65, color='steelblue', s=35)

limit = max(pairs_df['frame_npmi'].abs().max(), pairs_df['file_npmi'].abs().max()) * 1.1
ax.plot([-limit, limit], [-limit, limit], linestyle='--', color='gray', linewidth=1)
ax.axhline(0, color='lightgray', linewidth=0.8)
ax.axvline(0, color='lightgray', linewidth=0.8)
ax.set_xlim(-limit, limit)
ax.set_ylim(-limit, limit)
ax.set_xlabel('Frame-level NPMI')
ax.set_ylabel('File-level NPMI')
ax.set_title('Frame-level vs File-level NPMI per Class Pair')

for _, row in pairs_df.nlargest(10, 'gap').iterrows():
    ax.annotate(row['pair'], (row['frame_npmi'], row['file_npmi']), fontsize=7, xytext=(4, 4), textcoords='offset points')

plt.tight_layout()
plt.show()

## 3.1 Metadata distributions

The report's environment plot uses normalized environment labels, because the raw metadata contains many spelling variants and specific sublocations.


In [ ]:
SYNONYMS = {
    'toilet': 'bathroom',
    'livingroom': 'living_room',
    'washroom': 'bathroom',
    'restroom': 'bathroom',
    'work_place': 'office',
    'studyroom': 'office',
    'laboratory': 'office',
    'dorm_room': 'bedroom',
    'dorm_hallway': 'hallway',
    'apartment_hallway': 'hallway',
    'home_entryway': 'hallway',
    'house_entrance': 'hallway',
    'entrance': 'hallway',
    'entryway': 'hallway',
    'stairwell': 'hallway',
    'staircase': 'hallway',
    'lobby': 'hallway',
    'cloak_room': 'hallway',
    'home': 'living_room',
    'room': 'living_room',
    'apartment': 'living_room',
    'dining_room': 'kitchen',
    'cafeteria': 'kitchen',
    'pantry': 'kitchen',
    'laundry_room': 'bathroom',
    'outside': 'outdoor',
    'nature': 'outdoor',
    'parking_lot': 'outdoor',
    'sidewalk': 'outdoor',
    'driveway': 'outdoor',
    'porch': 'outdoor',
    'balcony': 'outdoor',
}

CORE_ENVIRONMENTS = {'kitchen', 'bedroom', 'living_room', 'hallway', 'office', 'bathroom', 'outdoor'}


def parse_environment(raw_value):
    tokens = re.split(r'[;,/]|_or_|_&_|\bor\b', str(raw_value).lower().strip())
    tokens = [token.strip().replace(' ', '_') for token in tokens if token.strip()]
    tokens = [SYNONYMS.get(token, token) for token in tokens]

    normalized = []
    for token in tokens:
        if token in CORE_ENVIRONMENTS:
            normalized.append(token)
            continue
        match = next((core for core in CORE_ENVIRONMENTS if core in token), None)
        if match is not None:
            normalized.append(match)

    unique = []
    for token in normalized:
        if token not in unique:
            unique.append(token)
    return unique


def primary_environment(raw_value):
    environments = parse_environment(raw_value)
    if not environments:
        return 'other'
    if len(environments) == 1:
        return environments[0]
    return 'mixed'


metadata['env_clean'] = metadata['recording_environment'].apply(primary_environment)

environment_counts = metadata['env_clean'].value_counts()
top_environments = environment_counts.head(7)

environment_summary = pd.DataFrame({
    'recordings': environment_counts,
    'share': (environment_counts / len(metadata)).round(3),
})
environment_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(top_environments.index, top_environments.values, color='steelblue')
for index, value in enumerate(top_environments.values):
    ax.text(index, value + 8, str(value), ha='center', va='bottom', fontsize=8)
ax.set_title('Recording Distribution by Environment (Top 7)')
ax.set_xlabel('Environment')
ax.set_ylabel('Number of recordings')
ax.set_xticklabels(top_environments.index, rotation=30, ha='right')
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '3a_Fig_Metadata_Environment.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
placement_counts = metadata['device_placement'].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(placement_counts.index, placement_counts.values, color=['steelblue', 'indianred'][:len(placement_counts)], width=0.4)
for index, value in enumerate(placement_counts.values):
    ax.text(index, value + 15, str(value), ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Number of Recordings')
ax.set_title('Recording Distribution by Device Placement')
ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '3a_Fig_Metadata_Placement.png', dpi=200, bbox_inches='tight')
plt.show()

placement_counts

## 3.2 Feature statistics

This version matches the report more closely: each feature family is first averaged per file from the stored `_mean` arrays, while the global range is taken from the stored `_min` and `_max` arrays.


In [ ]:
FEATURE_PREFIXES = {
    'ZCR': 'zcr',
    'Energy': 'energy',
    'Power': 'power',
    'Spectral Flux': 'flux',
    'Flatness': 'flatness',
    'Centroid': 'centroid',
    'Bandwidth': 'bandwidth',
    'Rolloff Low': 'rolloff_low',
    'Rolloff High': 'rolloff_high',
    'MFCC': 'mfcc',
    'MFCC delta': 'mfcc_d',
    'MFCC delta2': 'mfcc_d2',
    'Mel Spectrogram': 'melspect',
    'Contrast': 'contrast',
}

accumulated_features = {
    feature_name: {'means': [], 'mins': [], 'maxs': []}
    for feature_name in FEATURE_PREFIXES
}

for file_id in sorted(feature_file_map):
    data = load_feature_file(file_id)
    for feature_name, prefix in FEATURE_PREFIXES.items():
        mean_key = f'{prefix}_mean'
        min_key = f'{prefix}_min'
        max_key = f'{prefix}_max'
        if mean_key not in data or min_key not in data or max_key not in data:
            continue
        accumulated_features[feature_name]['means'].append(float(np.asarray(data[mean_key], dtype=float).mean()))
        accumulated_features[feature_name]['mins'].append(float(np.asarray(data[min_key], dtype=float).min()))
        accumulated_features[feature_name]['maxs'].append(float(np.asarray(data[max_key], dtype=float).max()))

feature_stats_rows = []
for feature_name, values in accumulated_features.items():
    per_file_means = np.asarray(values['means'], dtype=float)
    global_min = np.asarray(values['mins'], dtype=float).min()
    global_max = np.asarray(values['maxs'], dtype=float).max()
    feature_stats_rows.append({
        'Feature': feature_name,
        'Mean': per_file_means.mean(),
        'Std': per_file_means.std(),
        'Min': global_min,
        'Max': global_max,
        'Range': global_max - global_min,
    })

feature_stats_df = pd.DataFrame(feature_stats_rows).set_index('Feature')
feature_stats_df[['Mean', 'Std', 'Range']].round(3)

In [ ]:
names = list(feature_stats_df.index)
means = feature_stats_df['Mean'].values
stds = feature_stats_df['Std'].values
ranges = feature_stats_df['Range'].values
x = np.arange(len(names))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Audio Feature Statistics ({len(feature_file_map):,} files)', fontsize=13, fontweight='bold')

axes[0].barh(x, np.abs(means), xerr=stds, color='steelblue', alpha=0.8,
             error_kw={'ecolor': 'black', 'capsize': 3})
axes[0].set_yticks(x)
axes[0].set_yticklabels(names, fontsize=9)
axes[0].set_xlabel('|Mean| (absolute value)')
axes[0].set_title('(a) Mean +/- Std')
axes[0].set_xscale('log')
axes[0].xaxis.grid(True, linestyle='--', alpha=0.5)

colors = plt.cm.tab20(np.linspace(0, 1, len(names)))
axes[1].barh(x, ranges, color=colors, alpha=0.8)
axes[1].set_yticks(x)
axes[1].set_yticklabels(names, fontsize=9)
axes[1].set_xlabel('Range (max - min)')
axes[1].set_title('(b) Range')
axes[1].set_xscale('log')
axes[1].xaxis.grid(True, linestyle='--', alpha=0.5)

axes[2].scatter(ranges, stds, s=70, color=colors, zorder=3)
for index, name in enumerate(names):
    axes[2].annotate(name, (ranges[index], stds[index]), fontsize=7,
                     xytext=(4, 3), textcoords='offset points')
axes[2].set_xlabel('Range')
axes[2].set_ylabel('Std')
axes[2].set_xscale('log')
axes[2].set_yscale('log')
axes[2].set_title('(c) Std vs. Range')
axes[2].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '3b_Fig_Feature_Statistics.png', bbox_inches='tight', dpi=150)
plt.show()

## 3.3 Feature correlation

The report's Pearson heatmap is computed on frame-level feature-family means. Multi-dimensional features such as MFCC, Mel spectrogram, and contrast are averaged across coefficients per frame so every family contributes one comparable channel.


In [ ]:
import random

FEATURE_CATEGORY_KEYS = {
    'MFCC': 'mfcc_mean',
    'MFCC delta': 'mfcc_d_mean',
    'Mel Spect': 'melspect_mean',
    'Energy': 'energy_mean',
    'ZCR': 'zcr_mean',
    'Centroid': 'centroid_mean',
    'Flux': 'flux_mean',
    'Flatness': 'flatness_mean',
    'Contrast': 'contrast_mean',
}

random.seed(42)
sample_ids = random.sample(list(feature_file_map.keys()), min(300, len(feature_file_map)))

feature_category_rows = []
for file_id in sample_ids:
    data = load_feature_file(file_id)
    if any(key not in data for key in FEATURE_CATEGORY_KEYS.values()):
        continue
    frame_count = np.asarray(data['zcr_mean']).shape[0]
    frame_matrix = np.zeros((frame_count, len(FEATURE_CATEGORY_KEYS)))
    for column_index, key in enumerate(FEATURE_CATEGORY_KEYS.values()):
        values = np.asarray(data[key], dtype=float).reshape(frame_count, -1)
        frame_matrix[:, column_index] = values.mean(axis=1)
    feature_category_rows.append(frame_matrix)

feature_category_matrix = np.concatenate(feature_category_rows, axis=0)
print(f'Feature matrix: {feature_category_matrix.shape[0]:,} frames x {feature_category_matrix.shape[1]} feature categories')

In [ ]:
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import squareform

try:
    import seaborn as sns
except ImportError as exc:
    raise ImportError('Install seaborn to render the clustered feature-correlation heatmap.') from exc

feature_names = list(FEATURE_CATEGORY_KEYS.keys())
correlation_matrix = np.corrcoef(feature_category_matrix.T)

distance_matrix = 1 - np.abs(correlation_matrix)
np.fill_diagonal(distance_matrix, 0)
distance_matrix = (distance_matrix + distance_matrix.T) / 2
cluster_order = leaves_list(linkage(squareform(distance_matrix), method='average'))
correlation_ordered = correlation_matrix[np.ix_(cluster_order, cluster_order)]
feature_names_ordered = [feature_names[index] for index in cluster_order]

mask = np.triu(np.ones_like(correlation_ordered, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    correlation_ordered,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    vmin=-1,
    vmax=1,
    linewidths=0.4,
    linecolor='white',
    xticklabels=feature_names_ordered,
    yticklabels=feature_names_ordered,
    annot_kws={'size': 8.5},
    cbar_kws={'label': 'Pearson r', 'shrink': 0.75},
    ax=ax,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
ax.set_title('Feature Category Correlation\n(Pearson r, frame-level means)', fontsize=11, pad=10)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '3c_Fig_Feature_Category_Correlation.png', bbox_inches='tight', dpi=150)
plt.show()

pairs = [
    (correlation_matrix[i, j], feature_names[i], feature_names[j])
    for i in range(len(feature_names))
    for j in range(i + 1, len(feature_names))
]
pairs.sort(key=lambda item: abs(item[0]), reverse=True)
print('Top 5 strongest correlations:')
for value, feature_a, feature_b in pairs[:5]:
    print(f'  {feature_a} <-> {feature_b}: r = {value:.3f}')

## 3.4 Feature-space visualization with t-SNE

This is the report-aligned setup from the external notebook: each frame with exactly one active class and annotator agreement at least `0.70` becomes one data point. The feature vector has 76 dimensions: `MFCC x32`, `MFCC-delta x32`, `ZCR`, `Energy`, `Centroid`, `Flux`, `Flatness`, and `Contrast x7`.


In [ ]:
import time
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import RobustScaler, StandardScaler

FRAME_FEATURE_KEYS = [
    ('mfcc_mean', 32),
    ('mfcc_d_mean', 32),
    ('zcr_mean', 1),
    ('energy_mean', 1),
    ('centroid_mean', 1),
    ('flux_mean', 1),
    ('flatness_mean', 1),
    ('contrast_mean', 7),
]
AGREEMENT_THRESHOLD = 0.70

X_frames = []
y_frames = []

for file_id, labels in hard_labels_dict.items():
    labels = np.asarray(labels)
    data = load_feature_file(file_id)
    if 'annotations' not in data or data['annotations'].shape[2] < 2:
        continue
    if any(key not in data for key, _ in FRAME_FEATURE_KEYS):
        continue
    raw_annotations = data['annotations'].astype(float)
    for frame_index in range(labels.shape[0]):
        active = np.where(labels[frame_index] == 1)[0]
        if len(active) != 1:
            continue
        annotation_frame = raw_annotations[frame_index]
        intersection = ((annotation_frame[:, 0] > 0) & (annotation_frame[:, 1] > 0)).sum()
        union = ((annotation_frame[:, 0] > 0) | (annotation_frame[:, 1] > 0)).sum()
        if union == 0 or intersection / union < AGREEMENT_THRESHOLD:
            continue
        X_frames.append(np.concatenate([
            np.asarray(data[key][frame_index], dtype=float).reshape(-1)[:width]
            for key, width in FRAME_FEATURE_KEYS
        ]))
        y_frames.append(TARGET_CLASSES[active[0]])

X_frames = np.asarray(X_frames)
y_frames = np.asarray(y_frames)
print(f'Frame matrix: {X_frames.shape[0]:,} frames x {X_frames.shape[1]} dimensions')
print(f'Filter: exactly one active class, annotator agreement >= {AGREEMENT_THRESHOLD}')
pd.Series(y_frames).value_counts()

In [ ]:
import matplotlib.patheffects as path_effects

X_standardized = StandardScaler().fit_transform(X_frames)
X_clipped = np.clip(X_standardized, -5, 5)
X_robust = RobustScaler().fit_transform(X_clipped)

pca50 = PCA(n_components=50, random_state=42)
X_pca50 = pca50.fit_transform(X_robust)
X_pca2 = PCA(n_components=2, random_state=42).fit_transform(X_robust)
print(f'PCA-50 retains {pca50.explained_variance_ratio_.sum():.1%} of variance')

tsne_results = {}
for perplexity in [30, 50, 100]:
    start = time.time()
    tsne_results[perplexity] = TSNE(
        n_components=2,
        perplexity=perplexity,
        learning_rate='auto',
        init='pca',
        random_state=42,
        n_jobs=-1,
    ).fit_transform(X_pca50)
    print(f't-SNE perplexity={perplexity}: {time.time() - start:.0f}s')

COLORS = [
    '#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4',
    '#42d4f4', '#f032e6', '#bfef45', '#469990', '#9A6324',
    '#dcbeff', '#800000', '#aaffc3', '#808000', '#ffd8b1',
]
SHORT_CLASS_NAMES = {
    'keyboard_typing': 'keyboard',
    'keychain': 'keychain',
    'footsteps': 'footsteps',
    'door_open_close': 'door',
    'cutlery_dishes': 'cutlery',
    'toilet_flushing': 'toilet',
    'running_water': 'water',
    'light_switch': 'light sw.',
    'window_open_close': 'window',
    'bell_ringing': 'bell',
    'wardrobe_drawer_open_close': 'wardrobe',
    'microwave': 'microwave',
    'vacuum_cleaner': 'vacuum',
    'phone_ringing': 'phone',
    'coffee_machine': 'coffee',
}
class_colors = {class_name: COLORS[index] for index, class_name in enumerate(TARGET_CLASSES)}


def draw_embedding(ax, coordinates, title):
    for class_name in TARGET_CLASSES:
        class_mask = y_frames == class_name
        if not class_mask.any():
            continue
        ax.scatter(
            coordinates[class_mask, 0],
            coordinates[class_mask, 1],
            c=class_colors[class_name],
            s=4,
            alpha=0.25,
            linewidths=0,
            rasterized=True,
            label=SHORT_CLASS_NAMES[class_name],
        )
    for class_name in TARGET_CLASSES:
        class_mask = y_frames == class_name
        if not class_mask.any():
            continue
        center_x = coordinates[class_mask, 0].mean()
        center_y = coordinates[class_mask, 1].mean()
        ax.text(
            center_x,
            center_y,
            SHORT_CLASS_NAMES[class_name],
            fontsize=7,
            ha='center',
            va='center',
            fontweight='bold',
            color=class_colors[class_name],
            path_effects=[path_effects.withStroke(linewidth=2.5, foreground='white')],
        )
    ax.set_title(title, fontsize=10, pad=6)
    ax.set_xticks([])
    ax.set_yticks([])

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
draw_embedding(axes[0, 0], X_pca2, 'PCA (2D)')
draw_embedding(axes[0, 1], tsne_results[30], 't-SNE perplexity = 30')
draw_embedding(axes[1, 0], tsne_results[50], 't-SNE perplexity = 50')
draw_embedding(axes[1, 1], tsne_results[100], 't-SNE perplexity = 100')
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5, fontsize=8, frameon=True,
           bbox_to_anchor=(0.5, -0.03), markerscale=3, handletextpad=0.3)
fig.suptitle(
    'Feature Space - Frame-level embeddings '
    '(76 dimensions: MFCC x32, MFCC-delta x32, ZCR, Energy, Centroid, Flux, Flatness, Contrast x7)\n'
    f'{X_frames.shape[0]:,} frames, agreement >= {AGREEMENT_THRESHOLD}, exactly one active class per frame',
    fontsize=10,
    fontweight='bold',
)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(FIGURES_DIR / '3d_Fig_Feature_Space_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(10, 8))
draw_embedding(
    ax,
    tsne_results[50],
    f'Feature Space (t-SNE, perplexity=50)\nFrame-level, {X_frames.shape[0]:,} frames, agreement >= {AGREEMENT_THRESHOLD}',
)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8,
          frameon=True, markerscale=3, title='Class', title_fontsize=9)
ax.set_xlabel('t-SNE 1', fontsize=9)
ax.set_ylabel('t-SNE 2', fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '3d_Fig_Feature_Space_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

## Closing Notes

This notebook provides a reproducible companion to the Task 3 report. It walks through annotation agreement, majority-vote label aggregation, label co-occurrence, metadata bias, feature statistics, feature correlation, and feature-space visualization.

The main takeaway is that the dataset is usable but non-trivial: annotator agreement is generally good, class and environment distributions are imbalanced, and frame-level audio features show limited class separability. These findings motivate careful preprocessing, robust evaluation splits, and models that can use longer temporal context.
